# Brand Marketing Brainstorm: LangGraph + AutoGen with Human-in-the-Loop

This notebook keeps the structure for the tool-calling agent and plugs it into an iterative brainstorming flow.

**Two graphs, one inside the other**

```
OUTER GRAPH (brainstorm loop, state = BrainstormState)

START ──> run_tools ──> marketing team ──> human_review ──┬──> END        (you type "exit")
              ^                                           │
              └───────────────────────────────────────────┘  (you type a new direction)


INNER GRAPH (research agent from the course, state = AgentState)
used inside the run_tools node

START ──> agent ──should_continue──> action (ToolNode: search / extract)
            ^                            │
            └────────────────────────────┘      no tool call ──> END
```

* **Inner graph** (course pattern): the LLM receives the tools list, decides which tool to call (`agent` node), `ToolNode` executes it (`action` node), and the loop repeats until the LLM answers without a tool call. Adding a new tool means adding it to the list.
* **Outer graph**: runs the inner graph once per iteration (`run_tools`), sends the result to the AutoGen Brand Marketer (`brand_marketer`) and pauses for your feedback (`human_review`). Your feedback either starts a new iteration, with the history accumulated, or ends the session.

Cells marked **(course)** keep the structure you learned; comments explain what was fixed.

## 1. Install packages

* `pyautogen` is the legacy AutoGen 0.2 package. The import `autogen_agentchat` comes from `autogen-agentchat`, and the OpenAI-compatible model client comes from `autogen-ext[openai]`.
* `langchain-tavily` is the current Tavily integration. `langchain_community.tools.tavily_search.TavilySearchResults` is deprecated.

In [ ]:
!pip install -q -U langgraph langchain-openai langchain-tavily autogen-agentchat "autogen-ext[openai]" python-dotenv

## 2. Imports and API keys

In [ ]:
import os
import uuid
import operator
from typing import TypedDict, Annotated, Sequence, Literal

from IPython.display import display, Markdown, Image

# LangChain
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_tavily import TavilySearch, TavilyExtract

# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langgraph.errors import GraphRecursionError

# AutoGen (new API)
from autogen_agentchat.agents import AssistantAgent
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient

KEY_NAMES = ["OPENAI_API_KEY", "TAVILY_API_KEY", "GOOGLE_API_KEY"]
try:
    from google.colab import userdata   # only exists on Colab
    for name in KEY_NAMES:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass                        # secret not defined: fine if you don't use that provider
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

for name in KEY_NAMES:
    print(f"{name}: {'found' if os.environ.get(name) else 'MISSING'}")

In [ ]:
# Helper function to display markdown nicely (course)
def print_markdown(text):
    # Displays text as Markdown in Jupyter.
    display(Markdown(text))

## 3. Settings

* `MAX_ITERATIONS`: safety stop for the brainstorm loop.
* `RESEARCH_RECURSION_LIMIT`: safety stop for the inner research graph. In LangGraph each node execution is one "step"; `agent -> action -> agent -> action -> agent` is 5 steps, so a limit of 12 allows about 5 rounds of tool calls before LangGraph raises `GraphRecursionError`. This limits how much autonomy the research agent has.
* `MARKETER_PROVIDER`: `"openai"` or `"gemini"`.

In [ ]:
MAX_ITERATIONS = 5
RESEARCH_RECURSION_LIMIT = 12

MARKETER_PROVIDER = "openai"                # "openai" or "gemini"
OPENAI_MARKETER_MODEL = "gpt-5-mini"
GEMINI_MARKETER_MODEL = "gemini-2.5-flash"  # replace with the Gemini model your key has access to

EXIT_WORDS = {"exit", "quit", "stop", "end"}

## 4. Instantiating the LangChain LLM

This is the model that decides which tools to call. The gpt-5 family only accepts the default temperature.

In [ ]:
llm = ChatOpenAI(model="gpt-5-mini")
print("LangChain OpenAI Chat Model configured.")

## 5. The marketing agents team agent (AutoGen)


In [ ]:
if MARKETER_PROVIDER == "openai":
    marketer_client = OpenAIChatCompletionClient(
        model=OPENAI_MARKETER_MODEL,
        api_key=os.environ["OPENAI_API_KEY"],
    )
else:
    marketer_client = OpenAIChatCompletionClient(
        model=GEMINI_MARKETER_MODEL,
        api_key=os.environ["GOOGLE_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        model_info={"vision": False, "function_calling": True, "json_output": True,
                    "structured_output": True, "family": "unknown"},
    )

brand_marketer_agent = AssistantAgent(
    name="Brand_Marketer",
    model_client=marketer_client,
    system_message=(
        "You are an experienced brand marketer for company which manufactures niches, sinks and countertops made of porcelain tile flooring. The products have different colours and standard sizes and are destinated to bathrooms."
        "Attention: remember the company does not manufacture porcelain floors! It manufactures the bathroom niches, countertops and sinks from that material."
        "You receive the brainstorming history, the user's current direction and fresh web research. "
        "Produce actionable insights."
        "Cite the source URL next to each claim taken from the research, and separate facts from your hypotheses. "
        "End with 2 or 3 questions that would help the user steer the next iteration."
        "You should help the company create a strong brand in people's mind."
        "Think that the company is almost 30 years old, a very niched market, despite a good network within the Brazilian construction materials market (São Paulo and Minas Gerais states)."
        "The business model is majorly built upon confidence and personal contact with clients, and the company operates B2B, but wants to innovate in marketing strategies."
    )
)



cmo_agent = AssistantAgent(
    name="Chief_Marketing_Officer",
    model_client=marketer_client,
    system_message=(
        "You are an enthusiastic but rigorous senior Chief Marketing Officer, working for a company which manufactures niches, sinks and countertops made of porcelain tile flooring. The products have different colours and standard sizes and are destinated to bathrooms."
        "Attention: remember the company does not manufacture porcelain floors! It manufactures the bathroom niches, countertops and sinks from that material."
        "You receive the brainstorming history."
        "Produce actionable insights: positioning, target audience, messaging, channels and campaign ideas."
        "Consider we are in a small company and we should try effective and measurable results (limited resources)."
        " The company is almost 30 years old, a very niched market, despite a good network within the Brazilian construction materials market (São Paulo and Minas Gerais states)."
        "The campaigns should consider whether digital or physical channels. Think of creating a great proximity with the client and help the company to identify how to be remembered by its clients and potential clients."

    )
)


designer_agent = AssistantAgent(
    name="Market_Designer",
    model_client=marketer_client,
    system_message=(
        "You are a Mrketing Designer in a small company, which manufactures niches, sinks and countertops made of porcelain tile flooring. The products have different colours and standard sizes and are destinated to bathrooms."
        "Think of designs to be used in social media posts, based on what was provided by online sources or by human input (original pictures)."
        "Considering the company does not have any photoshop or editor, it needs written ideas or prompts to be used for images/posts generation."
        "If it is more adequate, give ideas/insights instead."
    )
)

print(f"Agent '{brand_marketer_agent.name, cmo_agent.name, designer_agent.name }' created ({MARKETER_PROVIDER}).")

## 6. The two states

* **`AgentState`** (course) belongs to the inner research graph. It only has `messages`; `operator.add` appends each new message (LLM answer, tool result) to the list. It is created fresh in every iteration, so tool calls and raw page texts of one round do not pile up in the next prompts.
* **`BrainstormState`** belongs to the outer graph and uses the fields you had planned: `actual_prompt`, `data_tools`, `finish`, plus one insight field per agent (`marketer_insight`, `cmo_insight`, `designer_insight`), so no agent overwrites another's answer. Two were added: `iteration` (counter) and `history`, which has the `operator.add` reducer so every node can *append* lines to it. Fields without a reducer are overwritten.

In [ ]:
# Inner graph state (course)
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]


# Outer graph state
class BrainstormState(TypedDict, total=False):
    actual_prompt: str                              # current direction (first prompt, then your feedback)
    data_tools: str                                 # research report of the current iteration
    marketer_insight: str                           # last Brand Marketer answer
    cmo_insight: str                                # last CMO answer
    designer_insight: str                           # last Marketing Designer answer
    finish: bool                                    # set by the human review node
    iteration: int
    history: Annotated[list[str], operator.add]     # accumulated across iterations

## 7. Tools

Instantiate each tool and put it in the list. The list is the only thing the rest of the code needs.

**To add a new tool:** write a function with the `@tool` decorator (the docstring is what the LLM reads to decide when to use it), then add it to `tools_list_single`.`bind_tools` shows it to the LLM and `ToolNode` knows how to run it. An example is commented below.

In [ ]:
tavily_search_tool = TavilySearch(max_results=3)
tavily_extract_tool = TavilyExtract(extract_depth="advanced")

# Example of a custom tool (uncomment and add it to the list to use it):
# '@tool'
# def word_count(text: str) -> int:
#     "Counts the words in a text."
#     return len(text.split())

tools_list_single = [tavily_search_tool, tavily_extract_tool]

## 8. Research agent node factory

The outer function receives the tools and returns the inner function, which is the node.



In [ ]:
def make_call_model_with_tools(tools: list):
    # Binds the tools to the language model (done once, when the node is built)
    model_with_tools = llm.bind_tools(tools)

    def call_model_with_tools(state: AgentState):
        print("DEBUG: Entering call_model_with_tools node")
        messages = state["messages"]

        # Feeds the conversation history (messages) into the model
        response = model_with_tools.invoke(messages)

        # Return the model response as a new message
        return {"messages": [response]}

    return call_model_with_tools

In [ ]:
# Conditional edge logic (course, unchanged)
def should_continue(state: AgentState) -> Literal["action", "__end__"]:
    # Determines the next step: continue with tools or end.
    print("DEBUG: Entering should_continue node")
    last_message = state["messages"][-1]

    if isinstance(last_message, AIMessage) and getattr(last_message, "tool_calls", None):
        print("DEBUG: Decision: continue (route to action)")
        return "action"
    print("DEBUG: Decision: end (route to END)")
    return END

In [ ]:
# Graph builder
def build_graph_one_tool(tools_list):
    tool_node = ToolNode(tools_list)                     # runs whatever tool the LLM asked for
    call_node_fn = make_call_model_with_tools(tools_list)

    graph_one_tool = StateGraph(AgentState)
    graph_one_tool.add_node("agent", call_node_fn)
    graph_one_tool.add_node("action", tool_node)
    graph_one_tool.set_entry_point("agent")
    graph_one_tool.add_conditional_edges("agent", should_continue, {"action": "action", END: END})
    graph_one_tool.add_edge("action", "agent")

    app = graph_one_tool.compile()

    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception as error:   # the PNG is rendered by an online service
        print("Could not render the graph image:", error)

    return app

In [ ]:
def app_call(app, messages):
    # Runs the research graph alone and prints every message (course, useful for testing)
    initial_state = {"messages": [HumanMessage(content=messages)]}
    final_state = app.invoke(initial_state)
    for i in final_state["messages"]:
        print_markdown(i.type)
        print_markdown(str(i.content))
        if i.additional_kwargs != {}:
            print(i.additional_kwargs)
    return final_state["messages"][-1].content, final_state

Build the research graph. The variable is called `research_app` (instead of `app`) because the outer graph will also be compiled, and two things called `app` would overwrite each other.

In [ ]:
research_app = build_graph_one_tool(tools_list_single)

Optional: test the research graph alone (uses API credits).

## 9. Nodes of the outer graph

**`node_run_tools`** runs the whole research graph as one step. It builds a fresh conversation (`SystemMessage` with the research instructions + `HumanMessage` with recent history and the current direction), invokes `research_app` and keeps only the final report (the last message) in `data_tools`. The names of the tools called go to `history`, so you can follow what the agent did.

**`node_brand_marketer`** is your original node, translated and with the fields aligned with the state. One addition: `on_reset()` before `run()`. AutoGen agents keep an internal memory between `run()` calls; since the full history already goes in the task, keeping that memory would send everything twice. With the reset, LangGraph's state is the single source of truth.

**`node_human_review`** is your `node_revisao_humana`. `interrupt(payload)` stops the graph and saves the state in the checkpointer; when resumed with `Command(resume="text")`, `interrupt` returns that text. On resume LangGraph re-runs the node **from its first line**, so nothing expensive should come before `interrupt`.

**Why the `UserProxyAgent` was not used:** calling `interrupt` from inside an AutoGen callback means that on resume LangGraph re-runs the whole AutoGen call, and AutoGen may catch the exception that `interrupt` uses internally. The human pause belongs to the flow, so it lives in a LangGraph node.

In [ ]:
RESEARCH_SYSTEM = (
    "You are a market research assistant for a marketing team of a small company."
    "The company is almost 30 years old, a very niched market, despite a good network within the Brazilian construction materials market (São Paulo and Minas Gerais states)."
    "Attention: remember the company does not manufacture porcelain floors! It manufactures the bathroom niches, countertops and sinks from that material."
    "Search in the languages most likely to return relevant sources for the target market: "
    "for a Brazilian market, run queries in Brazilian Portuguese and in English. "
    "Use the search tool to find recent and relevant information. "
    "Then use the extract tool on at most 3 of the most relevant URLs to read them in full. "
    "Finish with a concise research report: key facts, numbers, competitors and trends, "
    "each with its source URL. Do not invent information that is not in the tool results."
    "The web search shoud support the brainstorm to be conducted."
)


def node_run_tools(state: BrainstormState) -> dict:
    iteration = state.get("iteration", 0) + 1
    print(f"\n===== Iteration {iteration}: research =====")

    recent_context = "\n\n".join(state.get("history", [])[-6:]) or "(none)"
    messages = [
        SystemMessage(content=RESEARCH_SYSTEM),
        HumanMessage(content=(
            f"Previous brainstorm context:\n{recent_context}\n\n"
            f"Current direction:\n{state['actual_prompt']}\n\n"
            "Research this direction, looking for information not covered before."
        )),
    ]

    try:
        final_state = research_app.invoke({"messages": messages}, {"recursion_limit": RESEARCH_RECURSION_LIMIT})
        report = final_state["messages"][-1].content
        tool_calls = [
            f"{call['name']}({call['args']})"
            for message in final_state["messages"] if isinstance(message, AIMessage)
            for call in (message.tool_calls or [])
        ]
    except GraphRecursionError:
        report = "(research stopped: the agent reached the tool-call limit)"
        tool_calls = []

    return {
        "iteration": iteration,
        "data_tools": report,
        "history": [f"[Iteration {iteration}] Tools called: " + ("; ".join(tool_calls) or "none")],
    }


async def node_brand_marketer(state: BrainstormState) -> dict:
    print(f"===== Iteration {state['iteration']}: brand marketer =====")

    # Accumulated context
    complete_text = "\n\n".join(state.get("history", []))
    task = (
        f"Accumulated context so far:\n{complete_text}\n\n"
        f"New data collected by the tools:\n{state['data_tools']}\n\n"
        f"Current user direction: {state['actual_prompt']}\n"
        "Based on this, generate your Brand Marketing insights."
    )

    # Runs the AutoGen agent (stateless: everything it knows comes from `task`, storaged in report and produced by the Tavily tools)
    await brand_marketer_agent.on_reset(CancellationToken())
    response = await brand_marketer_agent.run(task=task)
    last_insight = response.messages[-1].content

    return {
        "marketer_insight": last_insight,
        "history": [f"[Iteration {state['iteration']}] Brand Marketer:\n{last_insight}"],
    }

async def node_cmo(state: BrainstormState) -> dict:
    print(f"===== Iteration {state['iteration']}: Chief Marketing Officer =====")

    # Accumulated context
    complete_text = "\n\n".join(state.get("history", []))
    task = (
        f"Accumulated context so far:\n{complete_text}\n\n"
        f"New data collected by the tools:\n{state['data_tools']}\n\n"
        f"Current user direction: {state['actual_prompt']}\n"
        "Based on this, generate your Chief Marketing Officer insights."
    )

    # Runs the AutoGen agent (stateless: everything it knows comes from `task`, storaged in report and produced by the Tavily tools)
    await cmo_agent.on_reset(CancellationToken())
    response = await cmo_agent.run(task=task)
    last_insight = response.messages[-1].content

    return {
        "cmo_insight": last_insight,
        "history": [f"[Iteration {state['iteration']}] CMO:\n{last_insight}"],
    }


async def node_designer(state: BrainstormState) -> dict:
    print(f"===== Iteration {state['iteration']}: Marketing Designer =====")

    # Accumulated context
    complete_text = "\n\n".join(state.get("history", []))
    task = (
        f"Accumulated context so far:\n{complete_text}\n\n"
        f"New data collected by the tools:\n{state['data_tools']}\n\n"
        f"Current user direction: {state['actual_prompt']}\n"
        "Based on this, generate your Marketing Designer insights."
    )

    # Runs the AutoGen agent (stateless: everything it knows comes from `task`, storaged in report and produced by the Tavily tools)
    await designer_agent.on_reset(CancellationToken())
    response = await designer_agent.run(task=task)
    last_insight = response.messages[-1].content

    return {
        "designer_insight": last_insight,
        "history": [f"[Iteration {state['iteration']}] Marketing Designer:\n{last_insight}"],
    }



def node_human_review(state: BrainstormState) -> dict:
    # Pauses the workflow and sends the insights for you to evaluate
    feedback = interrupt({
        "iteration": state["iteration"],
        "insight_to_evaluate": (                     # the three agents' answers of this iteration
            f"## Brand Marketer\n{state['marketer_insight']}\n\n"
            f"## Chief Marketing Officer\n{state['cmo_insight']}\n\n"
            f"## Marketing Designer\n{state['designer_insight']}"
        ),
        "instruction": "Type a new direction to iterate, or 'exit' to finish.",
    })
    user_text = str(feedback).strip()

    if user_text.lower() in EXIT_WORDS:
        return {"finish": True}

    # Otherwise, update actual_prompt with your new direction and save it in the history
    return {
        "finish": False,
        "actual_prompt": user_text,
        "history": [f"[Iteration {state['iteration'] + 1}] User direction: {user_text}"],
    }


def decide_next_step(state: BrainstormState) -> Literal["run_tools", "__end__"]:
    if state.get("finish"):
        return END
    if state["iteration"] >= MAX_ITERATIONS:
        print(f"Reached MAX_ITERATIONS={MAX_ITERATIONS}; stopping.")
        return END
    return "run_tools"

## 10. Building the outer graph

`MemorySaver` is the **checkpointer**: it saves the state after each step, which is what allows `interrupt` to pause and resume. It keeps everything in RAM; for persistence across sessions use `SqliteSaver`.

In [ ]:
workflow = StateGraph(BrainstormState)


workflow.add_node("run_tools", node_run_tools)
workflow.add_node("brand_marketer", node_brand_marketer)
workflow.add_node("cmo", node_cmo)
workflow.add_node("designer", node_designer)
workflow.add_node("human_review", node_human_review)

workflow.add_edge(START, "run_tools")
workflow.add_edge("run_tools", "brand_marketer")
workflow.add_edge("brand_marketer", "cmo")
workflow.add_edge("cmo", "designer")
workflow.add_edge("designer", "human_review")
workflow.add_conditional_edges("human_review", decide_next_step, {"run_tools": "run_tools", END: END})

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as error:
    print("Could not render the graph image:", error)

## 11. Driver loop

* `thread_id` identifies one brainstorm session in the checkpointer; a new id is a new session.
* `ainvoke` runs until the `interrupt` (async because `node_brand_marketer` is async; Colab accepts `await` in a cell).
* `aget_state` shows where the graph stopped: `snapshot.next` is empty when it reached `END`, and `snapshot.tasks[0].interrupts[0].value` is the payload passed to `interrupt`.
* `Command(resume=answer)` resumes the graph, delivering your answer to `interrupt`.

In [ ]:
async def run_brainstorm(first_prompt: str) -> dict:
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    initial_state = {
        "actual_prompt": first_prompt,
        "iteration": 0,
        "finish": False,
        "history": [f"[Iteration 1] User prompt: {first_prompt}"],
    }

    result = await app.ainvoke(initial_state, config)        # runs until the first pause

    while "__interrupt__" in result:                          # graph is paused
        payload = result["__interrupt__"][0].value
        print_markdown(payload["insight_to_evaluate"])
        answer = input("New direction or 'exit': ").strip()
        result = await app.ainvoke(Command(resume=answer), config)  # continues until the next pause or the END.
                                                                    #Refers to the outer loop (needs .ainvoke because the loop contains async functions)

    return result                                             # without "__interrupt__": END

## 12. Run it

In [ ]:
first_prompt = (
    "I want to prospect new trends in the Brazilian decoration/construction materials."
    "So if my porcelain-made products, niches, countertops and sinks are a good investment for the next year,"
    "and would like to analyze how the market is and what"
    "I can do to keep my business competitive. So, please, give me not only marketing campaigns, but also strategies."
)
final_state = await run_brainstorm(first_prompt) #Uses await because of the async function 'run_brainstorm'

In [ ]:
# Inspect the accumulated history
for entry in final_state["history"]:
    print(entry, "\n")